# Suite VLG — The Verilog language

Verilog and SystemVerilog as one tree language (`@mbse/programs/Verilog/Syntax`): its kinds and categories,
organized as IEEE 1800's grammar is, the identifiers it accepts, and where each kind and feature exists in the two
families, which the standards check.

In [ ]:
import { SystemVerilog2017, SystemVerilog2023, Syntax as S, Verilog2005, VerilogStandard } from "@mbse/programs/Verilog";

import { assert, equal } from "./support.js";

const L = S.LANGUAGE;
const [V1995, V2001] = [new VerilogStandard(1995, "Verilog"), new VerilogStandard(2001, "Verilog")];
const [SV2005, SV2009, SV2012] = [new VerilogStandard(2005), new VerilogStandard(2009), new VerilogStandard(2012)];
const name = (spelling: unknown) => new S.Identifier({ spelling });
const ref = (spelling: string) => new S.NameExpression({ name: name(spelling) });
const number = (spelling = "1") => new S.IntegerLiteral({ spelling });
const unit = (...items: any[]) => new S.SourceText({ items });
const module = (items: any[] = [], properties: object = {}) =>
  new S.ModuleDeclaration({ keyword: "module", name: name("m"), items, ...properties });

## VLG-01 · Every kind belongs to one category, and the grammar lists them

In [ ]:
const kinds = L.kinds();
assert(kinds.size === S.KINDS.length && S.KINDS.length === 275);
equal([...kinds.keys()], S.KINDS.map((k) => k.name));
equal([...L.categories().keys()].sort(), ["Name", "Item", "Port", "DataType", "Dimension", "Statement", "Connection",
  "TimingControl", "Expression", "Literal", "Range", "Directive", "Constraint", "Property", "Sequence", "BinsSelect", "ProductionItem", "Pattern"].sort());
assert(new S.IntegerLiteral() instanceof S.Literal && new S.IntegerLiteral() instanceof S.Expression);
assert(S.KINDS.every((k) => k.NAME === `Programs.Verilog.${k.KIND}`));
const grammar = L.grammar() as any;
equal(grammar.base, { Verilog: 1995, SystemVerilog: 2005 });
assert(grammar.kinds.length === 275);
equal([S.sv(), S.sv(2009), S.verilog(2001)], [{ SystemVerilog: 2005 }, { SystemVerilog: 2009 },
  { Verilog: 2001, SystemVerilog: 2005 }]);
assert(new S.Comment({ text: " x" }).block === false && new S.SourceText().items.length === 0);
console.log("ok");

## VLG-02 · Identifiers are simple or escaped, and system names start with `$`

In [ ]:
equal(L.validate(name("x_1$")), []);
equal(L.validate(name("\\bus+index")), []);
equal(L.validate(name("")), []);
for (const bad of ["1x", "$x", "a b", "Größe", "\\", "\\a b"]) {
  equal(L.validate(name(bad)), [`${bad === "\\" ? "'\\\\'" : bad === "\\a b" ? "'\\\\a b'" : `'${bad}'`} is not an identifier`]);
}
equal(L.validate(name(3n)), ["Identifier.spelling must be a str, got int"]);
equal(L.validate(new S.SystemCall({ name: "$clog2" })), []);
equal(L.validate(new S.SystemCall({ name: "" })), []);
for (const bad of ["clog2", "$", "$a b", "$é"]) {
  equal(L.validate(new S.SystemCall({ name: bad })), [`'${bad}' is not a system task or function name`]);
}
equal(L.validate(new S.SystemCall({ name: 1n })), ["SystemCall.name must be a str, got int"]);
console.log("ok");

## VLG-03 · Kinds and features exist by family and year: Verilog has none of SystemVerilog's own

In [ ]:
equal([Verilog2005.STANDARD.name(), SystemVerilog2017.STANDARD.name(), SystemVerilog2023.STANDARD.name(), V1995.name()],
  ["Verilog-2005", "SystemVerilog-2017", "SystemVerilog-2023", "Verilog-1995"]);
const pkg = new S.PackageDeclaration({ name: name("p") });
equal(Verilog2005.check(pkg), ["PackageDeclaration is not Verilog"]);
equal(SV2005.check(pkg), []);
const ansi = new S.AnsiPort({ direction: "input", name: name("a") });
equal(V1995.check(ansi), ["AnsiPort needs Verilog-2001"]);
equal(V2001.check(ansi), []);
equal(Verilog2005.check(new S.IntegerVectorType({ keyword: "logic" })), ["IntegerVectorType.keyword 'logic' is not Verilog"]);
equal(Verilog2005.check(new S.IntegerVectorType({ keyword: "reg", signing: "signed" })), []);
equal(V1995.check(new S.IntegerVectorType({ keyword: "reg", signing: "signed" })),
  ["IntegerVectorType.signing 'signed' needs Verilog-2001"]);
const unique0 = new S.IfStatement({ qualifier: "unique0", condition: ref("a"), consequence: new S.NullStatement() });
equal(SV2005.check(unique0), ["IfStatement.qualifier 'unique0' needs SystemVerilog-2009"]);
equal(SV2009.check(unique0), []);
const final = new S.ImmediateAssertion({ keyword: "assert", deferral: "final", expression: ref("a") });
equal(SV2009.check(final), ["ImmediateAssertion.deferral 'final' needs SystemVerilog-2012"]);
equal(SV2012.check(final), []);
const star = new S.EventControl();
equal(V1995.check(star), ["EventControl * needs Verilog-2001"]);
equal(V2001.check(star), []);
equal(Verilog2005.check(new S.DisableStatement()), ["DisableStatement fork is not Verilog"]);
equal(Verilog2005.check(new S.DisableStatement({ target: ref("b") })), []);
const declared = new S.ForStatement({ initializers: [new S.VariableDeclaration({
  type: new S.IntegerAtomType({ keyword: "integer" }),
  declarators: [new S.VariableDeclarator({ name: name("i"), value: number() })] })], body: new S.NullStatement() });
equal(Verilog2005.check(declared), ["ForStatement.initializers declarations is not Verilog"]);
const assign = (target: string) => new S.AssignmentExpression({ target: ref(target), operator: "=", value: number() });
const two = new S.ForStatement({ initializers: [assign("i"), assign("j")], body: new S.NullStatement() });
equal(Verilog2005.check(two), ["ForStatement.initializers declarations is not Verilog"]);
equal(SV2005.check(two), []);
equal(Verilog2005.check(new S.ForStatement({ initializers: [assign("i")], body: new S.NullStatement() })), []);
equal(Verilog2005.check(module([], { labeled: true })), ["ModuleDeclaration.labeled is not Verilog"]);
equal(V1995.check(new S.BinaryExpression({ left: number(), operator: "**", right: number() })),
  ["BinaryExpression.operator '**' needs Verilog-2001"]);
equal(SV2005.check(new S.BinaryExpression({ left: number(), operator: "<->", right: number() })),
  ["BinaryExpression.operator '<->' needs SystemVerilog-2009"]);
const interfaceClass = new S.ClassDeclaration({ interface: true, name: name("i") });
equal(Verilog2005.check(interfaceClass), ["ClassDeclaration is not Verilog", "ClassDeclaration.interface is not Verilog"]);
equal(SV2009.check(interfaceClass), ["ClassDeclaration.interface needs SystemVerilog-2012"]);
equal(SV2009.check(new S.ForwardTypedefDeclaration({ keyword: "interface class", name: name("i") })),
  ["ForwardTypedefDeclaration.keyword 'interface class' needs SystemVerilog-2012"]);
equal(Verilog2005.check(new S.VariableDeclaration({ random: "rand", declarators: [new S.VariableDeclarator({ name: name("r") })] })),
  ["VariableDeclaration.random is not Verilog"]);
equal(Verilog2005.check(new S.NewExpression()), ["NewExpression is not Verilog"]);
console.log("ok");

## VLG-04 · A standard checks a whole tree, by path

In [ ]:
const tree = unit(module([new S.AlwaysConstruct({ keyword: "always_ff", body: new S.NullStatement() }),
  new S.ContinuousAssign({ assignments: [new S.AssignmentExpression({ target: ref("a"), operator: "+=", value: number() })] })]));
equal(Verilog2005.check(tree), ["items[0].items[0]: AlwaysConstruct.keyword 'always_ff' is not Verilog",
  "items[0].items[1].assignments[0]: AssignmentExpression.operator '+=' is not Verilog"]);
equal(SystemVerilog2023.check(tree), []);
console.log("ok");

## VLG-05 · Classes and methods check their own shape

In [ ]:
const method = (kind: typeof S.FunctionDeclaration | typeof S.TaskDeclaration, properties: object = {}) =>
  new kind({ name: name("f"), ...properties });
for (const kind of [S.FunctionDeclaration, S.TaskDeclaration]) {
  equal(L.validate(method(kind, { virtual: true, pure: true })), []);
  equal(L.validate(method(kind, { extern: true })), []);
  equal(L.validate(method(kind, { pure: true })), [`a pure ${kind.KIND} is virtual`]);
  equal(L.validate(method(kind, { extern: true, body: [new S.NullStatement()] })), [`an extern or pure ${kind.KIND} has no body`]);
  equal(L.validate(method(kind, { virtual: true, pure: true, labeled: true })), [`an extern or pure ${kind.KIND} has no body`]);
}
const base = () => new S.NamedType({ name: name("b") });
equal(L.validate(new S.ClassDeclaration({ name: name("c"), base: base(), arguments: [number()] })), []);
equal(L.validate(new S.ClassDeclaration({ interface: true, name: name("i"), interfaces: [new S.NamedType({ name: name("j") })] })), []);
equal(L.validate(new S.ClassDeclaration({ interface: true, name: name("i"), base: base() })),
  ["an interface ClassDeclaration has no virtual, base or arguments"]);
equal(L.validate(new S.ClassDeclaration({ interface: true, virtual: true, name: name("i") })),
  ["an interface ClassDeclaration has no virtual, base or arguments"]);
equal(L.validate(new S.ClassDeclaration({ name: name("c"), arguments: [number()] })), ["a ClassDeclaration with arguments has a base"]);
console.log("ok");

## VLG-06 · Distributions check their own shape, and where constraints and randomization exist

In [ ]:
equal(L.validate(new S.DistItem({ value: number() })), []);
equal(L.validate(new S.DistItem({ value: number(), operator: ":=", weight: number() })), []);
for (const item of [new S.DistItem({ value: number(), operator: ":=" }), new S.DistItem({ value: number(), weight: number() })]) {
  equal(L.validate(item), ["a DistItem has both an operator and a weight, or neither"]);
}
equal(L.validate(new S.DistItem({ operator: ":=", weight: number() })), ["a default DistItem weighs its values with :/"]);
equal(L.validate(new S.DistItem()), ["a default DistItem weighs its values with :/"]);
equal(L.validate(new S.DistItem({ operator: ":/", weight: number() })), []);
equal(SystemVerilog2017.check(new S.DistItem({ operator: ":/", weight: number() })), ["DistItem default needs SystemVerilog-2023"]);
equal(SV2009.check(new S.ExpressionConstraint({ soft: true, expression: ref("a") })),
  ["ExpressionConstraint.soft needs SystemVerilog-2012"]);
equal(SV2009.check(new S.ConstraintPrototype({ qualifier: "pure", name: name("k") })),
  ["ConstraintPrototype.qualifier 'pure' needs SystemVerilog-2012"]);
equal(SV2009.check(new S.UniqueConstraint({ set: [ref("a")] })), ["UniqueConstraint needs SystemVerilog-2012"]);
equal(SV2005.check(new S.LocalName({ name: name("x") })), ["LocalName needs SystemVerilog-2009"]);
equal(SV2009.check(new S.RandomizeWithExpression({ call: ref("r"), restricted: true })),
  ["RandomizeWithExpression.restricted needs SystemVerilog-2012"]);
equal(Verilog2005.check(new S.RandCaseStatement()), ["RandCaseStatement is not Verilog"]);
console.log("ok");

## VLG-07 · Where assertions, their properties and clocking exist

In [ ]:
const restrict = new S.ConcurrentAssertion({ keyword: "restrict", spec: new S.PropertySpec({ property: ref("a") }) });
equal(SV2005.check(restrict), ["ConcurrentAssertion.keyword 'restrict' needs SystemVerilog-2009"]);
equal(SV2009.check(restrict), []);
equal(Verilog2005.check(restrict), ["ConcurrentAssertion is not Verilog", "ConcurrentAssertion.keyword 'restrict' is not Verilog",
  "spec: PropertySpec is not Verilog"]);
const since2009 = [new S.ImplicationProperty({ antecedent: ref("a"), operator: "#-#", consequent: ref("b") }),
  new S.BinaryProperty({ left: ref("a"), operator: "until", right: ref("b") }),
  new S.UnaryProperty({ operator: "always", operand: ref("a") }), new S.StrengthProperty({ keyword: "strong", sequence: ref("a") }),
  new S.AbortProperty({ keyword: "accept_on", condition: ref("a"), operand: ref("b") }), new S.CaseProperty({ expression: ref("a") }),
  new S.LetDeclaration({ name: name("l"), value: ref("a") }), new S.DefaultDisable({ condition: ref("a") }),
  new S.ClockingDeclaration({ scope: "global", clock: new S.EventControl() }), new S.AssertionPort({ local: true, name: name("p") }),
  new S.KeywordType({ keyword: "untyped" })];
for (const node of since2009) {
  const problems = SV2005.check(node);
  assert(problems.length > 0 && problems.every((p) => p.includes("needs SystemVerilog-2009")) && SV2009.check(node).length === 0,
    String(node));
}
equal(SV2005.check(new S.UnaryProperty({ operator: "not", operand: ref("a") })), []);
equal(SV2005.check(new S.ImplicationProperty({ antecedent: ref("a"), operator: "|->", consequent: ref("b") })), []);
console.log("ok");

## VLG-08 · Covergroups, coverpoints, bins and transitions check their own shape, and where they exist

In [ ]:
const clock = new S.EventControl({ events: [new S.EventExpression({ expression: ref("clk") })] });
equal(L.validate(new S.CovergroupDeclaration({ name: name("g"), clock, sample: new S.SampleFunction() })),
  ["a CovergroupDeclaration has a clock or a sample, not both"]);
equal(L.validate(new S.Coverpoint({ type: new S.IntegerAtomType({ keyword: "int" }), expression: ref("a") })),
  ["a Coverpoint with a type has a label"]);
equal(L.validate(new S.CoverageBins({ keyword: "bins", name: name("b"), size: number(), initializer: new S.BinsDefault() })),
  ["a CoverageBins with a size is an array"]);
equal(L.validate(new S.CoverageBins({ keyword: "bins", name: name("b"), array: true, size: number(),
  initializer: new S.BinsDefault() })), []);
equal(L.validate(new S.TransitionStep({ values: [number()], operator: "*" })),
  ["a TransitionStep has both an operator and a count, or neither"]);
equal(L.validate(new S.TransitionStep({ values: [number()], operator: "*", count: number() })), []);
equal(Verilog2005.check(new S.CovergroupDeclaration({ name: name("g") })), ["CovergroupDeclaration is not Verilog"]);
const since2012 = [new S.BinsExpression({ expression: ref("a") }), new S.FilteredBinsSelect({ select: ref("a"), filter: ref("b") }),
  new S.BinsValues({ values: [number()], filter: ref("item") }),
  new S.Coverpoint({ label: name("c"), type: new S.IntegerAtomType({ keyword: "int" }), expression: ref("a") })];
for (const node of since2012) {
  const problems = SV2009.check(node);
  assert(problems.length > 0 && problems.every((p) => p.includes("needs SystemVerilog-2012")) && SV2012.check(node).length === 0,
    String(node));
}
equal(SV2005.check(new S.SampleFunction()), ["SampleFunction needs SystemVerilog-2009"]);
console.log("ok");

## VLG-09 · Attributes exist from Verilog-2001

In [ ]:
const keep = [new S.AttributeInstance({ specs: [new S.AttributeSpec({ name: name("keep") })] })];
const keptWire = new S.NetDeclaration({ net_type: "wire", declarators: [new S.VariableDeclarator({ name: name("w") })] });
const attributedWire = new S.AttributedItem({ attributes: keep, item: keptWire });
equal(V1995.check(attributedWire), ["AttributedItem needs Verilog-2001", "attributes[0]: AttributeInstance needs Verilog-2001",
  "attributes[0].specs[0]: AttributeSpec needs Verilog-2001"]);
equal(V2001.check(attributedWire), []);
for (const node of [new S.BinaryExpression({ left: ref("a"), operator: "+", attributes: [new S.AttributeInstance()], right: ref("b") }),
  new S.UnaryExpression({ operator: "-", attributes: [new S.AttributeInstance()], operand: ref("a") }),
  new S.ConditionalExpression({ condition: ref("a"), attributes: [new S.AttributeInstance()], consequence: ref("b"),
    alternative: ref("c") }),
  new S.CallExpression({ callee: ref("f"), attributes: [new S.AttributeInstance()] }),
  new S.SystemCall({ name: "$f", attributes: [new S.AttributeInstance()] })]) {
  const problems = V1995.check(node);
  assert(problems.includes(`${node.kind().KIND}.attributes needs Verilog-2001`) && V2001.check(node).length === 0,
    problems.join("; "));
}
console.log("ok");

## VLG-10 · Streaming, delays, patterns, tolerance ranges and the names and types SystemVerilog adds check their own shape, and where they exist

In [ ]:
const [ea, eb, ec] = [ref("a"), ref("b"), ref("c")];
equal(L.validate(new S.DelayControl({ value: ea, fall: eb, turnoff: ec })), []);
equal(L.validate(new S.DelayControl({ value: ref("a"), fall: ref("b") })), []);
equal(L.validate(new S.DelayControl({ value: ref("a"), turnoff: ref("c") })), ["a DelayControl with a turnoff has a fall"]);
const repeated = new S.AssignmentPattern({ count: number("2"), items: [ref("a"), ref("b")] });
equal(L.validate(repeated), []);
equal(SV2005.check(repeated), []);
equal(L.validate(new S.AssignmentPattern({ count: number("2"), items: [new S.PatternItem({ key: ref("a"), value: ref("b") })] })),
  ["a repeated AssignmentPattern has positional items only"]);
for (const item of [new S.StreamItem({ expression: ref("a") }), new S.StreamItem({ expression: ref("a"), left: ref("b") }),
  new S.StreamItem({ expression: ref("a"), left: ref("b"), operator: "+:", right: ref("c") })]) {
  equal(L.validate(item), []);
}
for (const item of [new S.StreamItem({ expression: ref("a"), left: ref("b"), operator: ":" }),
  new S.StreamItem({ expression: ref("a"), left: ref("b"), right: ref("c") }),
  new S.StreamItem({ expression: ref("a"), operator: ":", right: ref("c") })]) {
  equal(L.validate(item), ["a StreamItem's `with` has a left, and an operator and a right, or neither"]);
}
const stream = new S.StreamingConcatenation({ operator: "<<", slice: new S.IntegerAtomType({ keyword: "byte" }),
  items: [new S.StreamItem({ expression: ref("a"), left: ref("b") })] });
equal(SV2005.check(stream), ["items[0]: StreamItem.left needs SystemVerilog-2009"]);
equal(SV2009.check(stream), []);
equal(V2001.check(stream), ["StreamingConcatenation is not Verilog", "slice: IntegerAtomType.keyword 'byte' is not Verilog",
  "items[0]: StreamItem is not Verilog", "items[0]: StreamItem.left is not Verilog"]);
const tolerance = new S.ValueRange({ left: ref("a"), operator: "+/-", right: ref("b") });
equal(SystemVerilog2017.check(tolerance), ["ValueRange.operator needs SystemVerilog-2023"]);
equal(SystemVerilog2023.check(tolerance), []);
equal(V1995.check(new S.MinTypMaxExpression({ min: ref("a"), typ: ref("b"), max: ref("c") })), []);
equal(V1995.check(new S.EmptyArgument()), []);
for (const node of [new S.RootExpression(), new S.EmptyQueue(), new S.UnitName(), new S.TypeReference({ operand: ref("a") })]) {
  equal(V2001.check(node), [`${node.kind().KIND} is not Verilog`]);
  equal(SV2005.check(node), []);
}
const unitScoped = new S.ScopedName({ scope: new S.UnitName(), name: name("x") });
equal(L.validate(unitScoped), []);
equal(V2001.check(unitScoped), ["ScopedName is not Verilog", "scope: UnitName is not Verilog"]);
console.log("ok");


## VLG-11 · Strengths, net types, aliases, `defparam`, time units, enumerations' ranges, randomized members, `const` and `static` ports and type restrictions check their own shape, and where they exist

In [ ]:
const strong = new S.DriveStrength({ first: "strong0", second: "weak1" });
equal(L.validate(strong), []);
equal(L.validate(new S.DriveStrength({ first: "pull1", second: "highz0" })), []);
equal(L.validate(new S.DriveStrength({ first: "strong0", second: "weak0" })), ["a DriveStrength has a strength of 0 and one of 1"]);
equal(L.validate(new S.DriveStrength({ first: "highz0", second: "highz1" })), ["a DriveStrength is not highz for both"]);
equal(L.validate(new S.DriveStrength({ first: "strong0" })), ["a DriveStrength needs a second"]);
const strongNet = new S.NetDeclaration({ net_type: "wire", strength: strong, expansion: "vectored",
  declarators: [new S.VariableDeclarator({ name: name("w") })] });
equal(L.validate(strongNet), []);
equal(V1995.check(strongNet), []);
const charged = new S.NetDeclaration({ net_type: "wire", strength: new S.ChargeStrength({ size: "small" }),
  declarators: [new S.VariableDeclarator({ name: name("t") })] });
equal(L.validate(charged), ["a NetDeclaration with a charge strength is a trireg"]);
charged.net_type = "trireg";
equal(L.validate(charged), []);
equal(V1995.check(charged), []);
equal(V1995.check(new S.ContinuousAssign({ strength: new S.DriveStrength({ first: "supply1", second: "pull0" }) })), []);
equal(V1995.check(new S.DefParam({ assignments: [new S.DefParamAssignment({ target: ref("u"), value: number() })] })), []);
const nettype = new S.NetTypeDeclaration({ type: new S.NonIntegerType({ keyword: "real" }), name: name("t"), function: name("f") });
equal(SV2009.check(nettype), ["NetTypeDeclaration needs SystemVerilog-2012"]);
equal(SV2012.check(nettype), []);
equal(L.validate(new S.NetAlias({ nets: [ref("a")] })), ["a NetAlias has two nets or more"]);
equal(V2001.check(new S.NetAlias({ nets: [ref("a"), ref("b")] })), ["NetAlias is not Verilog"]);
const units = new S.TimeUnitsDeclaration({ keyword: "timeunit", time: new S.TimeLiteral({ spelling: "1ns" }),
  precision: new S.TimeLiteral({ spelling: "1ps" }) });
equal(L.validate(units), []);
equal(SV2005.check(units), ["TimeUnitsDeclaration.precision needs SystemVerilog-2009"]);
units.keyword = "timeprecision";
equal(L.validate(units), ["a timeprecision TimeUnitsDeclaration has no precision"]);
equal(L.validate(new S.EnumMember({ name: name("A"), left: number("1"), right: number("3") })), []);
equal(L.validate(new S.EnumMember({ name: name("A"), right: number("3") })), ["an EnumMember with a right has a left"]);
equal(SV2005.check(new S.StructMember({ random: "rand", type: new S.IntegerAtomType({ keyword: "int" }) })), []);
for (const port of [new S.TfPort({ const: true, direction: "ref", name: name("a") }),
  new S.TfPort({ direction: "ref", static: true, name: name("a") })]) {
  equal(L.validate(port), []);
  port.direction = "input";
  equal(L.validate(port), ["a const or static TfPort is a ref"]);
}
const staticPort = new S.TfPort({ direction: "ref", static: true, name: name("a") });
equal(SystemVerilog2017.check(staticPort), ["TfPort.static needs SystemVerilog-2023"]);
equal(SystemVerilog2023.check(staticPort), []);
equal(SV2005.check(new S.TfPort({ const: true, direction: "ref", name: name("a") })), []);
const restricted = new S.TypeParameterDeclaration({ keyword: "parameter", restriction: "enum",
  assignments: [new S.TypeAssignment({ name: name("T") })] });
equal(SystemVerilog2017.check(restricted), ["TypeParameterDeclaration.restriction needs SystemVerilog-2023"]);
console.log("ok");


## VLG-12 · Port references, explicit ports and modports' ports check their own shape, and where they exist

In [ ]:
for (const port of [new S.PortReference({ name: name("w") }), new S.PortReference({ name: name("w"), left: number("0") }),
  new S.PortReference({ name: name("w"), left: number("3"), operator: ":", right: number("0") })]) {
  equal(L.validate(port), []);
  equal(V1995.check(port), []);
}
equal(L.validate(new S.PortReference({ name: name("w"), left: number("3"), operator: ":" })),
  ["a PortReference's select has a left, and an operator and a right, or neither"]);
const nonAnsi = [new S.PortConcatenation({ references: [new S.PortReference({ name: name("a") })] }), new S.EmptyPort(),
  new S.ExplicitPort({ name: name("p"), value: new S.PortReference({ name: name("a") }) })];
assert(nonAnsi.every((p) => V1995.check(p).length === 0));
const explicitAnsi = new S.ExplicitAnsiPort({ direction: "input", name: name("p"), value: ref("x") });
equal(L.validate(explicitAnsi), []);
equal(V2001.check(explicitAnsi), ["ExplicitAnsiPort is not Verilog"]);
equal(SV2005.check(explicitAnsi), []);
equal(L.validate(new S.ModportPort({ direction: "input", explicit: true, name: name("a"), value: ref("b") })), []);
equal(L.validate(new S.ModportPort({ direction: "input", name: name("a"), value: ref("b") })),
  ["a ModportPort with a value is explicit"]);
equal(L.validate(new S.ModportSubroutine({ keyword: "import", name: name("f") })), []);
equal(L.validate(new S.ModportSubroutine({ keyword: "export" })), ["a ModportSubroutine has a name or a prototype"]);
equal(L.validate(new S.ModportSubroutine({ keyword: "import", name: name("f"), prototype: new S.TaskDeclaration({ name: name("f") }) })),
  ["a ModportSubroutine has a name or a prototype"]);
equal(SV2005.check(new S.ModportClocking({ name: name("cb") })), []);
console.log("ok");


## VLG-13 · Class, method and constraint specifiers, default arguments, empty items, interfaces' types, skipped loop variables and `ifdef conditions check their own shape, and where they exist

In [ ]:
const finalClass = new S.ClassDeclaration({ final: true, name: name("c"), base: new S.NamedType({ name: name("b") }), defaulted: true });
equal(L.validate(finalClass), []);
equal(SystemVerilog2017.check(finalClass), ["ClassDeclaration.final needs SystemVerilog-2023",
  "ClassDeclaration.defaulted needs SystemVerilog-2023"]);
equal(L.validate(new S.ClassDeclaration({ name: name("c"), defaulted: true })),
  ["a ClassDeclaration with default arguments has a base and no others"]);
equal(L.validate(new S.ClassDeclaration({ name: name("c"), base: new S.NamedType({ name: name("b") }), arguments: [number()],
  defaulted: true })), ["a ClassDeclaration with default arguments has a base and no others"]);
for (const node of [new S.FunctionDeclaration({ virtual: true, specifier: "initial", final: true, name: name("f") }),
  new S.TaskDeclaration({ virtual: true, specifier: "extends", name: name("t") }),
  new S.ConstraintDeclaration({ specifier: "extends", final: true, name: name("k") }),
  new S.ConstraintPrototype({ final: true, name: name("k") })]) {
  const problems = SystemVerilog2017.check(node);
  assert(problems.length > 0 && problems.every((p) => p.endsWith("needs SystemVerilog-2023")), problems.join("; "));
  equal(SystemVerilog2023.check(node), []);
}
equal(SV2005.check(new S.EmptyItem()), []);
equal(V2001.check(new S.EmptyItem()), ["EmptyItem is not Verilog"]);
equal(SV2005.check(new S.TypedefDeclaration({ visibility: "local", type: new S.IntegerAtomType({ keyword: "int" }), name: name("t") })), []);
equal(V2001.check(new S.InterfaceTypeName({ interface: name("bus"), name: name("t") })), ["InterfaceTypeName is not Verilog"]);
equal(L.validate(new S.ForeachStatement({ array: ref("q"), variables: [new S.EmptyArgument(), name("j")], body: new S.NullStatement() })), []);
const macroCondition = new S.ParenthesizedExpression({ expression: new S.BinaryExpression({
  left: ref("A"), operator: "&&", right: new S.UnaryExpression({ operator: "!", operand: ref("B") }) }) });
const conditional = new S.IfdefDirective({ condition: macroCondition });
equal(L.validate(conditional), []);
equal(SystemVerilog2017.check(conditional), ["IfdefDirective.condition needs SystemVerilog-2023"]);
equal(L.validate(new S.IfdefDirective()), ["an IfdefDirective has a name or a condition"]);
equal(L.validate(new S.ElsifDirective({ name: name("A"), condition: new S.ParenthesizedExpression({ expression: ref("B") }) })),
  ["an ElsifDirective has a name or a condition"]);
equal(L.validate(new S.ElsifDirective({ condition: ref("A") })), ["an ElsifDirective's condition is in parentheses"]);
for (const wrong of [new S.BinaryExpression({ left: ref("A"), operator: "+", right: ref("B") }), number(),
  new S.UnaryExpression({ operator: "-", operand: ref("A") })]) {
  equal(L.validate(new S.IfdefDirective({ condition: new S.ParenthesizedExpression({ expression: wrong }) })),
    ["an IfdefDirective's condition has macros' names, !, &&, ||, -> and <-> only"]);
}
console.log("ok");


## VLG-14 · Procedural continuous assignments, `wait fork`, `wait_order`, timed triggers, repeated event controls and elaboration tasks, and where they exist

In [ ]:
equal(L.validate(new S.EventTrigger({ nonblocking: true, timing: new S.DelayControl({ value: number() }), event: ref("e") })), []);
equal(L.validate(new S.EventTrigger({ timing: new S.DelayControl({ value: number() }), event: ref("e") })),
  ["an EventTrigger with a timing is nonblocking"]);
for (const node of [new S.ForceStatement({ keyword: "force", target: ref("a"), value: number() }),
  new S.ReleaseStatement({ keyword: "deassign", target: ref("a") }),
  new S.RepeatEventControl({ count: number("2"), event: new S.EventControl({ events: [new S.EventExpression({ expression: ref("e") })] }) })]) {
  equal(V1995.check(node), []);
}
for (const node of [new S.WaitForkStatement(), new S.WaitOrderStatement({ events: [ref("a"), ref("b")] })]) {
  equal(V2001.check(node), [`${node.kind().KIND} is not Verilog`]);
  equal(SV2005.check(node), []);
}
const elaboration = new S.ElaborationTask({ name: "$error", arguments: [new S.StringLiteral({ text: "bad" }), new S.EmptyArgument()] });
equal(L.validate(elaboration), []);
equal(SV2005.check(elaboration), ["ElaborationTask needs SystemVerilog-2009"]);
equal(SV2009.check(elaboration), []);
console.log("ok");


## VLG-15 · DPI imports and exports, `bind`, `extern` design units and `.*` ports check their own shape, and where they exist

In [ ]:
const dpiPrototype = new S.FunctionDeclaration({ type: new S.IntegerAtomType({ keyword: "int" }), name: name("f") });
const dpi = new S.DpiImport({ spec: "DPI-C", property: "pure", prototype: dpiPrototype });
equal(L.validate(dpi), []);
equal(SV2005.check(dpi), ["DpiImport.spec 'DPI-C' needs SystemVerilog-2009"]);
equal(SV2005.check(new S.DpiImport({ spec: "DPI", prototype: new S.TaskDeclaration({ name: name("t") }) })), []);
equal(L.validate(new S.DpiImport({ spec: "DPI", property: "pure", prototype: new S.TaskDeclaration({ name: name("t") }) })),
  ["a pure DpiImport is a function"]);
equal(V2001.check(new S.DpiExport({ spec: "DPI", keyword: "task", name: name("t") })), ["DpiExport is not Verilog"]);
const bound = new S.ModuleInstantiation({ module: name("mon"), instances: [new S.Instance({ name: name("i") })] });
equal(SV2005.check(new S.BindDirective({ target: ref("m"), instances: [ref("u")], instantiation: bound })), []);
const externModule = module([], { extern: true });
equal(L.validate(externModule), []);
equal(V2001.check(externModule), ["ModuleDeclaration.extern is not Verilog"]);
equal(L.validate(module([], { extern: true, labeled: true })), ["an extern ModuleDeclaration has no items"]);
equal(L.validate(new S.InterfaceDeclaration({ extern: true, name: name("i"), items: [new S.EmptyItem()] })),
  ["an extern InterfaceDeclaration has no items"]);
equal(L.validate(module([], { ports: [new S.WildcardPort()] })), []);
equal(L.validate(new S.ProgramDeclaration({ name: name("p"), ports: [new S.WildcardPort(), new S.PortReference({ name: name("a") })] })),
  ["a ProgramDeclaration's .* is its only port"]);
console.log("ok");


## VLG-16 · `randsequence` rules check their own shape, and where productions exist

In [ ]:
equal(L.validate(new S.ProductionRule({ items: [new S.ProductionCall({ name: name("a") })], weight: number("2"),
  code: new S.ProductionCode() })), []);
equal(L.validate(new S.ProductionRule({ items: [new S.ProductionCall({ name: name("a") })], bias: number() })),
  ["a ProductionRule with a bias is a rand join"]);
equal(L.validate(new S.ProductionRule({ rand_join: true, bias: number(), items: [new S.ProductionCall({ name: name("a") })] })), []);
equal(L.validate(new S.ProductionRule({ items: [new S.ProductionCall({ name: name("a") })], code: new S.ProductionCode() })),
  ["a ProductionRule's code follows its weight"]);
const randSequence = new S.RandSequenceStatement({ first: name("main"), productions: [new S.Production({ name: name("main"), rules: [
  new S.ProductionRule({ items: [new S.ProductionIf({ condition: ref("c"), consequence: new S.ProductionCall({ name: name("a") }) }),
    new S.ProductionRepeat({ count: number("2"), item: new S.ProductionCall({ name: name("b") }) }),
    new S.ProductionCase({ expression: ref("d"), items: [new S.ProductionCaseItem({ item: new S.ProductionCall({ name: name("e") }) })] })] })] })] });
equal(L.validate(randSequence), []);
equal(SV2005.check(randSequence), []);
equal(V2001.check(randSequence)[0], "RandSequenceStatement is not Verilog");
assert(new S.ProductionCall() instanceof S.ProductionItem && L.categories().has("ProductionItem"));
console.log("ok");


## VLG-17 · Checkers, their instances in procedures, package exports and nets of user-defined types check their own shape, and where they exist

In [ ]:
const checker = new S.CheckerDeclaration({ name: name("c"), ports: [new S.AssertionPort({ direction: "input", name: name("a") })],
  items: [new S.VariableDeclaration({ random: "rand", type: new S.IntegerVectorType({ keyword: "bit" }),
    declarators: [new S.VariableDeclarator({ name: name("r") })] })] });
equal(L.validate(checker), []);
equal(SV2005.check(checker), ["CheckerDeclaration needs SystemVerilog-2009"]);
equal(SV2009.check(checker), []);
const checkerInstance = new S.CheckerStatement({ instantiation: new S.ModuleInstantiation({
  module: new S.ScopedName({ scope: name("p"), name: name("c") }), instances: [new S.Instance({ name: name("i") })] }) });
equal(L.validate(checkerInstance), []);
equal(SV2009.check(checkerInstance), []);
equal(SV2005.check(checkerInstance), ["CheckerStatement needs SystemVerilog-2009"]);
const exportAll = new S.ExportDeclaration({ all: true });
equal(L.validate(exportAll), []);
equal(SV2005.check(exportAll), ["ExportDeclaration needs SystemVerilog-2009"]);
equal(L.validate(new S.ExportDeclaration()), ["an ExportDeclaration exports all or items"]);
equal(L.validate(new S.ExportDeclaration({ all: true, items: [new S.ImportItem({ package: name("p") })] })),
  ["an ExportDeclaration exports all or items"]);
const userNet = new S.NetDeclaration({ type: new S.NamedType({ name: name("nt") }), delay: new S.DelayControl({ value: number() }),
  declarators: [new S.VariableDeclarator({ name: name("w") })] });
equal(L.validate(userNet), []);
userNet.expansion = "vectored";
equal(L.validate(userNet), ["a NetDeclaration without a net type has a user-defined one, a NamedType, alone"]);
equal(L.validate(new S.NetDeclaration({ type: new S.IntegerVectorType({ keyword: "logic" }),
  declarators: [new S.VariableDeclarator({ name: name("w") })] })), ["a NetDeclaration without a net type has a user-defined one, a NamedType, alone"]);
console.log("ok");


## VLG-18 · Tagged and soft unions, tagged values, patterns, predicates and `case matches` check their own shape, and where they exist

In [ ]:
const taggedUnion = new S.StructType({ keyword: "union", qualifier: "tagged", members: [new S.StructMember({
  type: new S.KeywordType({ keyword: "void" }), declarators: [new S.VariableDeclarator({ name: name("none") })] })] });
equal(L.validate(taggedUnion), []);
equal(SV2005.check(taggedUnion), []);
taggedUnion.qualifier = "soft";
equal(SystemVerilog2017.check(taggedUnion), ["StructType.qualifier 'soft' needs SystemVerilog-2023"]);
equal(L.validate(new S.StructType({ keyword: "struct", qualifier: "tagged" })), ["a StructType with a qualifier is a union"]);
const matching = new S.MatchesExpression({ value: ref("x"), pattern: new S.TaggedPattern({ member: name("a"),
  pattern: new S.VariablePattern({ name: name("v") }) }) });
const predicate = new S.PredicateExpression({ conditions: [matching, ref("ok")] });
equal(L.validate(predicate), []);
equal(V2001.check(predicate)[0], "PredicateExpression is not Verilog");
equal(L.validate(new S.PredicateExpression({ conditions: [ref("a")] })), ["a PredicateExpression has two conditions or more"]);
equal(L.validate(new S.StructurePattern({ items: [new S.WildcardPattern(), new S.PatternMember({ name: name("f"), pattern: number() })] })),
  ["a StructurePattern's members are in order or by name, not both"]);
const patternItem = new S.PatternCaseItem({ pattern: new S.WildcardPattern(), guard: ref("ok"), body: new S.NullStatement() });
const matchingCase = new S.CaseStatement({ keyword: "case", expression: ref("x"), matches: true,
  items: [patternItem, new S.CaseItem({ body: new S.NullStatement() })] });
equal(L.validate(matchingCase), []);
equal(V2001.check(new S.TaggedExpression({ member: name("a") })), ["TaggedExpression is not Verilog"]);
matchingCase.inside = true;
equal(L.validate(matchingCase), ["a CaseStatement is inside or matches, not both"]);
[matchingCase.inside, matchingCase.matches] = [false, false];
equal(L.validate(matchingCase), ["a CaseStatement with PatternCaseItems matches"]);
[matchingCase.matches, matchingCase.items] = [true, [new S.CaseItem({ expressions: [number()], body: new S.NullStatement() })]];
equal(L.validate(matchingCase), ["a matching CaseStatement has PatternCaseItems and a default"]);
assert(L.categories().has("Pattern") && new S.VariablePattern() instanceof S.Pattern);
console.log("ok");


## VLG-19 · Gates, pulls and user-defined primitives check their own shape, and where they exist

In [ ]:
const andGate = new S.GateInstantiation({ keyword: "and", strength: new S.DriveStrength({ first: "strong0", second: "weak1" }),
  delay: new S.DelayControl({ value: number() }), instances: [new S.GateInstance({ terminals: [ref("o"), ref("a")] })] });
equal(L.validate(andGate), []);
equal(V1995.check(andGate), []);
const pullGate = new S.GateInstantiation({ keyword: "pullup", strength: new S.PullStrength({ strength: "weak1" }),
  instances: [new S.GateInstance({ terminals: [ref("p")] })] });
equal(L.validate(pullGate), []);
pullGate.keyword = "buf";
equal(L.validate(pullGate), ["a GateInstantiation with a pull strength is a pullup or a pulldown"]);
equal(L.validate(new S.GateInstantiation({ keyword: "and", primitive: name("u") })), ["a GateInstantiation is a gate's or a primitive's"]);
equal(L.validate(new S.GateInstance({ dimensions: [new S.RangeDimension({ left: number(), right: number("0") })] })),
  ["a GateInstance with dimensions has a name"]);
equal(L.validate(new S.UdpEntry({ inputs: "(01) 1 r *", current: "?", output: "-" })), []);
equal(L.validate(new S.UdpEntry()), ["an UdpEntry needs an inputs", "an UdpEntry needs an output"]);
equal(L.validate(new S.UdpEntry({ inputs: "(0z) q", output: "1" })), ["a UdpEntry's inputs are levels and edges"]);
equal(L.validate(new S.UdpEntry({ inputs: "0", current: "r", output: "1" })), ["a UdpEntry's current state is a level"]);
equal(L.validate(new S.UdpEntry({ inputs: "0", output: "-" })), ["a UdpEntry's output is 0, 1 or x, or - with a current state"]);
equal(L.validate(new S.UdpPort({ direction: "input", reg: true, names: [name("a")] })), ["an input UdpPort has no reg and no value"]);
equal(L.validate(new S.UdpPort({ names: [name("q")] })), ["a UdpPort without a direction is a reg"]);
equal(L.validate(new S.UdpPort({ direction: "output", names: [name("q")], value: number() })), ["a UdpPort with a value is one output reg"]);
const labeledUdp = new S.UdpDeclaration({ name: name("p"), ports: [name("q")], labeled: true });
equal(V2001.check(labeledUdp), ["UdpDeclaration.labeled is not Verilog"]);
equal(SV2005.check(labeledUdp), []);
console.log("ok");


## VLG-20 · Module paths, timing checks, pulse styles and specify parameters check their own shape, and where they exist

In [ ]:
const modulePath = new S.PathDeclaration({ inputs: [ref("a")], operator: "=>", outputs: [ref("y")], delays: [number()] });
equal(L.validate(modulePath), []);
equal(V1995.check(modulePath), []);
[modulePath.ifnone, modulePath.condition] = [true, ref("b")];
equal(L.validate(modulePath), ["an ifnone PathDeclaration has no condition"]);
equal(L.validate(new S.PathDeclaration({ inputs: [ref("a")], operator: "=>", outputs: [ref("y")], data: ref("d"), delays: [number()] })),
  ["a PathDeclaration's data has a polarity, and a polarity data"]);
equal(L.validate(new S.PathDeclaration({ inputs: [ref("a"), ref("b")], operator: "=>", outputs: [ref("y")], delays: [number()] })),
  ["a parallel PathDeclaration has one input and one output"]);
equal(L.validate(new S.PathDeclaration({ inputs: [ref("a"), ref("b")], operator: "*>", outputs: [ref("y")], delays: [number()] })), []);
const checkEvent = new S.TimingCheckEvent({ edge: "edge", descriptors: "01, 10", terminal: ref("clk") });
const timingCheck = new S.TimingCheck({ name: "$period", arguments: [checkEvent, number("10")] });
equal(L.validate(timingCheck), []);
equal(V1995.check(timingCheck), []);
checkEvent.edge = "posedge";
equal(L.validate(timingCheck), ["arguments[0]: a TimingCheckEvent's descriptors are an edge's"]);
const pulseStyle = new S.PulseStyleDeclaration({ keyword: "showcancelled", outputs: [ref("y")] });
equal(V1995.check(pulseStyle), ["PulseStyleDeclaration needs Verilog-2001"]);
equal(V2001.check(pulseStyle), []);
const specparam = new S.SpecparamDeclaration({ assignments: [new S.SpecparamAssignment({ name: name("PATHPULSE$a$y"),
  value: number(), limit: number("2") })] });
equal(L.validate(new S.SpecifyBlock({ items: [specparam] })), []);
console.log("ok");


## VLG-21 · Covergroup extensions, block events, `matches` selects, `super.new(default)`, interface port declarations, attributed connections and sequence arguments check their own shape, and where they exist

In [ ]:
const extendedGroup = new S.CovergroupDeclaration({ extends: true, name: name("g") });
equal(L.validate(extendedGroup), []);
equal(SystemVerilog2017.check(extendedGroup), ["CovergroupDeclaration.extends needs SystemVerilog-2023"]);
extendedGroup.ports = [new S.TfPort({ name: name("p") })];
equal(L.validate(extendedGroup), ["an extends CovergroupDeclaration has no ports"]);
const blockEvents = new S.BlockEventControl({ events: [new S.BlockEvent({ keyword: "begin", name: ref("f") }),
  new S.BlockEvent({ keyword: "end", name: ref("f") })] });
equal(L.validate(new S.CovergroupDeclaration({ name: name("g"), clock: blockEvents })), []);
equal(V2001.check(blockEvents), ["BlockEventControl is not Verilog", "events[0]: BlockEvent is not Verilog",
  "events[1]: BlockEvent is not Verilog"]);
const matchingSelect = new S.MatchesBinsSelect({ select: ref("x"), count: number("2") });
equal(L.validate(matchingSelect), []);
equal(SV2009.check(matchingSelect), ["MatchesBinsSelect needs SystemVerilog-2012"]);
const defaultedNew = new S.NewExpression({ scope: new S.SuperExpression(), defaulted: true });
equal(L.validate(defaultedNew), []);
equal(SystemVerilog2017.check(defaultedNew), ["NewExpression.defaulted needs SystemVerilog-2023"]);
equal(L.validate(new S.NewExpression({ defaulted: true })), ["a defaulted NewExpression is super.new(default)"]);
equal(L.validate(new S.NewExpression({ scope: new S.SuperExpression(), arguments: [number()], defaulted: true })),
  ["a defaulted NewExpression is super.new(default)"]);
const interfacePort = new S.InterfacePortDeclaration({ interface: name("bus_if"), modport: name("mp"),
  declarators: [new S.VariableDeclarator({ name: name("a") })] });
equal(L.validate(interfacePort), []);
equal(V2001.check(interfacePort), ["InterfacePortDeclaration is not Verilog"]);
const attributedConnection = new S.AttributedConnection({ attributes: [new S.AttributeInstance()], connection: ref("x") });
equal(V1995.check(attributedConnection), ["AttributedConnection needs Verilog-2001", "attributes[0]: AttributeInstance needs Verilog-2001"]);
equal(L.validate(new S.CallExpression({ callee: ref("p"), arguments: [new S.DelaySequence({ first: ref("a"),
  steps: [new S.DelayStep({ delay: number(), sequence: ref("b") })] })] })), []);
console.log("ok");
